(page-wavelets)=
# Time-causal wavelets

Standard wavelets are excellent at signal processing but work offline: they need
the whole signal before they can tell you anything about the beginning of it.
Streams -- audio, biosignals, sensor traces -- need *causal* wavelets, whose output
at time $t$ depends only on samples up to $t$. Signals arriving at unpredictable
rates additionally benefit from *scale covariance*: the response transforms
predictably when time is rescaled.

Norse provides the time-causal wavelets of {cite}`pedersen2026` as both a plain
transform and a spiking encoder. The spiking variants double as event-driven
analog-to-digital converters: they turn a continuous input straight into spike
trains, skipping the uniform sampler, and they export to
[NIR](https://neuroir.org/) for deployment on neuromorphic hardware.

| Wavelet | Causal | Recovery | Cost |
| --- | :---: | --- | --- |
| `DifferenceOfTimeCausalKernels` | yes | exact | cascade of $N$ integrators per scale |
| `DifferenceOfExponentials` | yes | exact | one integrator per scale |
| `SpikingDifferenceOfTimeCausalKernels` | yes | quantised, delayed | + one neuron pair per channel |
| `SpikingDifferenceOfExponentials` | yes | quantised, delayed | lightest streaming variant |


In [ ]:
import matplotlib.pyplot as plt
import torch
import norse.torch as norse

plt.style.use("../resources/matplotlibrc")


## Analysis and synthesis

Every wavelet has the same two methods: calling it analyses a signal into
per-channel coefficients, and `reconstruct` puts them back together.

With `n_channels = K + 1`, the first $K$ channels carry the bandpass differences
between adjacent smoothing scales and the last carries the lowpass residual at the
coarsest scale:

$$\Delta L_1 = L_1 - f, \qquad \Delta L_k = L_k - L_{k-1}, \qquad x_K = L_K,$$

taking the raw signal as the implicit finest level $L_0 = f$. Reconstruction is the
telescoping sum $f = x_K - \sum_k \Delta L_k$, so it is exact.


In [ ]:
dt = 1e-3
t = torch.arange(0, 1.0, dt)
signal = torch.sin(2 * torch.pi * 3 * t) + 0.4 * torch.sin(2 * torch.pi * 31 * t)

wavelet = norse.DifferenceOfTimeCausalKernels(n_channels=5, dt=dt, mu_max=0.05)
coefficients = wavelet(signal)   # (T, n_channels)
reconstruction = wavelet.reconstruct(coefficients)

print(f"scales (s): {[round(m, 5) for m in wavelet.time_scales().tolist()]}")
print(f"cascade depths: {wavelet.cascade_depths}")
print(f"reconstruction error: {(reconstruction - signal).abs().max():.2e}")


The two components separate by scale: the 31 Hz term appears in the fine
channels and the 3 Hz term in the coarse ones.


In [ ]:
fig, axes = plt.subplots(wavelet.n_channels + 1, 1, figsize=(9, 9), sharex=True)
axes[0].plot(t, signal, color="black")
axes[0].set_ylabel("input")
for k in range(wavelet.n_channels):
    axes[k + 1].plot(t, coefficients[:, k])
    label = "lowpass" if k == wavelet.n_channels - 1 else f"$\\Delta L_{k + 1}$"
    axes[k + 1].set_ylabel(label)
axes[-1].set_xlabel("time (s)")
fig.tight_layout()


### Causality

Perturbing one sample leaves every earlier output untouched, which is what lets
these filters run on a stream.


In [ ]:
perturbed = signal.clone()
perturbed[500] += 3.0
before, after = wavelet(signal), wavelet(perturbed)

print(f"max change before t=0.5 s: {(before[:500] - after[:500]).abs().max():.2e}")
print(f"max change after  t=0.5 s: {(before[500:] - after[500:]).abs().max():.2e}")


## Spiking variants

The spiking encoders replace coefficients with spike trains. A threshold fires only
on positive crossings while the bandpass channels are signed, so each channel drives
a *pair* of neurons -- one with $+x_k$, one with $-x_k$ -- and the output has
`2 * n_channels` rows ordered `[pos_0, neg_0, pos_1, neg_1, ...]`.

Channels are divided by their kernels' $L_2$ norms first, so one threshold value
means the same thing at every scale. The default threshold scales with each
channel's effective time constant.


In [ ]:
spiking = norse.SpikingDifferenceOfExponentials(n_channels=4, dt=dt, mu_max=0.05)
spikes = spiking(signal)   # (T, 2 * n_channels)

print(f"spike trains: {tuple(spikes.shape)}")
print(f"thresholds: {[round(v, 3) for v in spiking.threshold.tolist()]}")
print(f"total spikes: {int(spikes.sum())}")
print(f"density: {spikes.mean():.1%} of samples carry a spike")


In [ ]:
fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
bottom.plot(t, signal, color="black")
bottom.set_xlabel("time (s)")
bottom.set_ylabel("input")
time_idx, neuron = torch.where(spikes > 0)
top.scatter(t[time_idx], neuron, s=2, marker="|")
top.set_ylabel("neuron")
top.set_yticks(range(spikes.shape[-1]))
fig.tight_layout()


Raising the threshold trades spikes for fidelity, which is the knob these encoders
exist to expose. The other knob is `mu_mem`, the membrane time constant: at `dt`
the membrane settles within a sample, so the neuron compares rather than
integrates and saturates at one spike per step. It defaults to `5 * dt`, which
integrates over a few samples. Broadband or fast-changing input wants a shorter
membrane, slow stationary input a longer one.


In [ ]:
for threshold in [0.1, 0.5, 2.0]:
    encoder = norse.SpikingDifferenceOfExponentials(
        n_channels=4, dt=dt, mu_max=0.05, threshold=torch.tensor(threshold)
    )
    z = encoder(signal)
    print(f"threshold {threshold:<5} {int(z.sum()):>5} spikes   {z.mean():.1%} density")


### Recovering the signal from spikes

Synthesis is not the analysis filters run backwards. What the spikes encode is the
*membrane*, so `decode` inverts that integration to recover each channel's
coefficient, and the combination is then handed to the non-spiking transform's own
`reconstruct` -- the same telescoping sum as the exact inverse.

Each spike accounts for $\theta$ of membrane charge, which arrived as
$(1-\alpha)x$, so $\hat x = \theta (h_{\mathrm{mem}} * s) / (1 - \alpha)$.


In [ ]:
coefficients_true = spiking.analysis(signal)
coefficients_decoded = spiking.decode(spikes).detach()

for k in range(spiking.n_channels):
    corr = torch.corrcoef(
        torch.stack([coefficients_true[:, k], coefficients_decoded[:, k]])
    )[0, 1]
    print(f"channel {k}: coefficient correlation {corr:.3f}")


The estimate is biased low in amplitude, because a spike discharges the membrane
from $v \ge \theta$ rather than from exactly $\theta$. The default weights are the
unit weights of the exact telescoping sum, which need nothing fitted.

`fit_oracle_weights` instead solves for the weights by least squares against a
known signal. It is an **oracle**: it needs the clean signal the spikes came from,
which a deployed encoder does not have, and the solution is specific to that
signal rather than a fixed per-channel gain. Read it as an upper bound on what any
linear decoder can recover from these spike trains.


In [ ]:
def quality(model, x):
    recon = model.reconstruct(model(x)).detach()
    corr = torch.corrcoef(torch.stack([x, recon]))[0, 1]
    nrmse = (recon - x).pow(2).mean().sqrt() / x.std()
    return recon, corr, nrmse

_, corr, nrmse = quality(spiking, signal)
print(f"default weights: correlation {corr:.3f}, nRMSE {nrmse:.3f}")

spiking.fit_oracle_weights(signal)
recon, corr, nrmse = quality(spiking, signal)
print(f"oracle weights:  correlation {corr:.3f}, nRMSE {nrmse:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(t, signal, color="black", label="input")
ax.plot(t, recon, label="from spikes")
ax.set_xlabel("time (s)")
ax.legend()
fig.tight_layout()


## As an encoding stage in a pipeline

Nothing above is needed to *use* the encoder: analysis and thresholding are the
forward pass, and decoding is a separate concern. Shapes follow Norse's convention
-- time first, channels last -- so the encoder composes directly with the rest of
the library, turning a continuous signal into the spike trains a spiking network
consumes.

Gradients pass through the threshold's surrogate, so the encoder's own parameters
train with the network when `learn_threshold=True`.


In [ ]:
batch = torch.stack(
    [torch.sin(2 * torch.pi * f * t) for f in (3.0, 5.0, 7.0, 11.0)], dim=1
)   # (T, batch)

model = norse.SequentialState(
    norse.SpikingDifferenceOfExponentials(
        n_channels=4, dt=dt, mu_max=0.05, learn_threshold=True
    ),
    torch.nn.Linear(8, 16),
    norse.LIFCell(),
    torch.nn.Linear(16, 3),
)

out, _ = model(batch)
print(f"{tuple(batch.shape)} -> {tuple(out.shape)}   (T, batch, classes)")

out.sum().backward()
trained = [n for n, p in model.named_parameters() if p.grad is not None]
print(f"parameters receiving gradient: {len(trained)}")


## Exporting to NIR

Both variants export as a single linear chain, which is what most neuromorphic
backends can ingest:

```text
non-spiking: Input -> fanout Affine -> LI bank -> connectivity Affine -> Output
spiking:     Input -> fanout Affine -> LI bank -> connectivity Affine -> LIF -> Output
```

The fanout `Affine` broadcasts the scalar input across the lanes of the `LI` bank.
The bank has one lane per smoothing scale plus one extra that reproduces the raw
signal, which the leading bandpass channel $\Delta L_1 = L_1 - f$ needs. The
connectivity `Affine` then wires each output row with $\pm 1$ weights between
adjacent scales, and the lowpass row taps the coarsest lane directly.


In [ ]:
import nir

for name, model in [
    ("non-spiking", norse.DifferenceOfExponentials(n_channels=4, dt=dt, mu_max=0.05)),
    ("spiking", norse.SpikingDifferenceOfExponentials(n_channels=4, dt=dt, mu_max=0.05)),
]:
    graph = model.to_nir()
    width = graph.nodes["output"].output_type["output"][0]
    print(f"{name}: {width} outputs")
    for node_name, node in graph.nodes.items():
        print(f"    {node_name:14s} {type(node).__name__}")


`nir.write` serialises the graph for a hardware backend, and
{py:func}`norse.torch.utils.import_nir.from_nir` reads one back into Norse.


In [ ]:
graph = norse.SpikingDifferenceOfExponentials(
    n_channels=4, dt=dt, mu_max=0.05
).to_nir()
nir.write("spiking_doe.nir", graph)

reloaded = nir.read("spiking_doe.nir")
print(f"wrote and reloaded {len(reloaded.nodes)} nodes")


## Choosing the parameters

- `mu_max` is the coarsest smoothing scale in seconds, so it sets the lowest
  frequency the bank resolves. Pick it from the slowest structure you care about.
- `c > 1` is the ratio between adjacent scales. $\sqrt 2$ gives a dense bank,
  $2$ an octave spacing with fewer channels for the same range.
- `n_channels` then fixes how far down from `mu_max` the bank reaches: the finest
  scale is $\mu_{\max} c^{-(K-1)}$.
- `cascade_depth_max` trades cost for fidelity to the time-causal limit kernel.
  `DifferenceOfExponentials` pins it at 1, which is the cheapest useful choice and
  the one that maps most directly onto a bank of leaky integrators.

A scale $\mu$ smaller than a few samples cannot support a deep cascade, so the
depth actually used per channel is clamped; `wavelet.cascade_depths` reports it.

```{bibliography}
:filter: docname in docnames
```
